In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## 1. Chargement

In [2]:
df = pd.read_csv("project_dataset.csv", sep=";")
print(f"Nombre de lignes initiales : {len(df)}")
df.head()

Nombre de lignes initiales : 12070


,Date,Service,Gare de départ,Gare d'arrivée,Durée moyenne du trajet,Nombre de circulations prévues,Nombre de trains annulés,Commentaire annulations,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,Retard moyen de tous les trains au départ,Commentaire retards au départ,Nombre de trains en retard à l'arrivée,Retard moyen des trains en retard à l'arrivée,Retard moyen de tous les trains à l'arrivée,Commentaire retards à l'arrivée,Nombre trains en retard > 15min,Retard moyen trains en retard > 15 (si liaison concurrencée par vol),Nombre trains en retard > 30min,Nombre trains en retard > 60min,Prct retard pour causes externes,Prct retard pour cause infrastructure,Prct retard pour cause gestion trafic,Prct retard pour cause matériel roulant,Prct retard pour cause gestion en gare et réutilisation de matériel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"
0,2018-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141.0,870,5.0,NaN,289.0,11.24780854,3.693179191,NaN,147.0,28.43673469,6.511117534,NaN,110.0,6.51,44.0,8.0,36.13445378,31.09243697,10.92436975,15.96638655,"5,04",0.840336134
1,2018-01,National,LE MANS,PARIS MONTPARNASSE,56.0,406.0,1.0,NaN,213.0,8.479968701,4.567119342,NaN,105.0,18.049,5.363539095,"Ce mois-ci, l'OD a été touchée par les inciden...",32.0,5.363539095,9.0,4.0,20.0,35.0,16.66666667,16.66666667,8.333333333,3.333333333
2,2018-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166.0,226.0,0.0,NaN,21.0,6.23968254,0.286283186,NaN,19.0,24.73684211,2.938053097,NaN,11.0,2.938053097,6.0,1.0,22.22222222,27.77777778,16.66666667,16.66666667,5.555555556,11.11111111
3,2018-01,National,PARIS MONTPARNASSE,NANTES,216.21,508.0,3.0,NaN,71.0,7.235211268,0.98,NaN,58.0,33.72643678,5.292211221,NaN,39.0,5.292211221,18.0,NaN,33.33333333,22.22222222,16.66666667,20.37037037,5.555555556,1.851851852
4,2018-01,National,POITIERS,PARIS MONTPARNASSE,94.0,472.0,4.0,NaN,224.0,6.784672619,3.229700855,NaN,89.0,14.59269663,4.882371795,NaN,42.0,4.882371795,10.0,0.0,15.78947368,45.61403509,NaN,15.78947368,1.754385965,1.754385965


## 2. Filtrage des lignes parasites

In [3]:
# On garde uniquement les lignes dont la date commence par un chiffre
df = df[df['Date'].astype(str).str.match(r'^\d', na=False)].copy()
print(f"Lignes après filtrage des commentaires : {len(df)}")

Lignes après filtrage des commentaires : 12010


## 3. Standardisation des colonnes texte

In [4]:
text_cols = ["Gare de départ", "Gare d'arrivée", "Service"]
for col in text_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.upper()
            .str.replace(r'\s+', ' ', regex=True)
        )

## 4. Suppression des doublons

In [5]:
initial_count = len(df)
df = df.drop_duplicates()
print(f"Doublons supprimés : {initial_count - len(df)}")
print(f"Lignes restantes   : {len(df)}")

Doublons supprimés : 214
Lignes restantes   : 11796


## 5. Conversion numérique

In [6]:
numeric_cols = [
    "Durée moyenne du trajet", "Nombre de circulations prévues", "Nombre de trains annulés",
    "Nombre de trains en retard au départ", "Retard moyen des trains en retard au départ",
    "Retard moyen de tous les trains au départ", "Nombre de trains en retard à l'arrivée",
    "Retard moyen des trains en retard à l'arrivée", "Retard moyen de tous les trains à l'arrivée",
    "Nombre trains en retard > 15min", "Retard moyen trains en retard > 15 (si liaison concurrencée par vol)",
    "Nombre trains en retard > 30min", "Nombre trains en retard > 60min",
    "Prct retard pour causes externes", "Prct retard pour cause infrastructure",
    "Prct retard pour cause gestion trafic", "Prct retard pour cause matériel roulant",
    "Prct retard pour cause gestion en gare et réutilisation de matériel",
    "Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"
]

numeric_cols = [c for c in numeric_cols if c in df.columns]

for col in numeric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(',', '.', regex=False)
        .str.strip()
        .replace({'-': np.nan, '': np.nan, 'nan': np.nan, 'None': np.nan})
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Conversion numérique terminée.")
nan_counts = df[numeric_cols].isnull().sum()
print(f"NaN dans les colonnes numériques :\n{nan_counts[nan_counts > 0]}")

Conversion numérique terminée.
NaN dans les colonnes numériques :
Durée moyenne du trajet                                                                        301
Nombre de circulations prévues                                                                 236
Nombre de trains annulés                                                                       236
Nombre de trains en retard au départ                                                           235
Retard moyen des trains en retard au départ                                                    293
Retard moyen de tous les trains au départ                                                      291
Nombre de trains en retard à l'arrivée                                                         235
Retard moyen des trains en retard à l'arrivée                                                  299
Retard moyen de tous les trains à l'arrivée                                                    292
Nombre trains en retard > 15min            

## 6. Conversion de la date

In [7]:
df["Date"] = pd.to_datetime(df["Date"], format="mixed", dayfirst=False, errors="coerce")

nb_dates_invalides = df["Date"].isnull().sum()
if nb_dates_invalides > 0:
    print(f"{nb_dates_invalides} lignes avec date invalide supprimées.")
df = df.dropna(subset=['Date'])

df["Année"] = df["Date"].dt.year.astype(int)
df["Mois"]  = df["Date"].dt.month.astype(int)

print(f"Plage temporelle : {df['Date'].min().date()} → {df['Date'].max().date()}")

Plage temporelle : 2018-01-01 → 2025-12-01


## 7. Typage final

In [8]:
df = df.astype({col: 'string' for col in df.select_dtypes(include='object').columns})
print(df.dtypes)

Date                                                                                           datetime64[ns]
Service                                                                                        string[python]
Gare de départ                                                                                 string[python]
Gare d'arrivée                                                                                 string[python]
Durée moyenne du trajet                                                                               float64
Nombre de circulations prévues                                                                        float64
Nombre de trains annulés                                                                              float64
Commentaire annulations                                                                        string[python]
Nombre de trains en retard au départ                                                                  float64
Retard moy

## 8. Gestion des valeurs manquantes

In [9]:
# Colonnes numériques : on remplace par 0
df[numeric_cols] = df[numeric_cols].fillna(0)

comment_cols = [
    "Commentaire annulations",
    "Commentaire retards au départ",
    "Commentaire retards à l'arrivée"
]
for col in comment_cols:
    if col in df.columns:
        df[col] = df[col].fillna(pd.NA)
        df[col] = df[col].fillna("Non communiqué")

# Colonnes catégorielles
categorical_cols = ["Service", "Gare de départ", "Gare d'arrivée"]
_invalid_values = ["", "nan", "none", "NAN", "NONE", "NAT", "0", "INCONNU", "<NA>"]

for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].str.strip()
        df[col] = df[col].replace(_invalid_values, pd.NA)
        df[col] = df[col].fillna("Inconnu")

# Suppression des lignes sans Service
initial_len = len(df)
df = df[df["Service"] != "Inconnu"]
print(f"Lignes supprimées car 'Service' manquant : {initial_len - len(df)}")

# Suppression des lignes sans gare
initial_len = len(df)
df = df[(df["Gare de départ"] != "Inconnu") & (df["Gare d'arrivée"] != "Inconnu")]
print(f"Lignes supprimées car gare manquante : {initial_len - len(df)}")

print(f"\nValeurs manquantes restantes : {df.isnull().sum().sum()}")

Lignes supprimées car 'Service' manquant : 232
Lignes supprimées car gare manquante : 130

Valeurs manquantes restantes : 0


## 9. NOUVEAU — Corrections qualité

### 9.1 Uniformisation des commentaires

In [10]:
# NC, -, <NA> → "Non communiqué" pour uniformité
comment_replacements = ["-", "NC", "<NA>", "nan", ""]

for col in comment_cols:
    if col in df.columns:
        df[col] = df[col].replace(comment_replacements, "Non communiqué")

for col in comment_cols:
    print(f"\n{col}:")
    print(df[col].value_counts().head(3))


Commentaire annulations:
Commentaire annulations
Non communiqué    11434
Name: count, dtype: Int64

Commentaire retards au départ:
Commentaire retards au départ
Non communiqué    11434
Name: count, dtype: Int64

Commentaire retards à l'arrivée:
Commentaire retards à l'arrivée
Non communiqué                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

### 9.2 Valeurs négatives impossibles → 0

In [11]:
# Les retards moyens et les comptages ne peuvent pas être négatifs
# Les valeurs négatives sont des erreurs de saisie ou d'encodage = remplacées par 0
cols_non_negative = [
    "Retard moyen de tous les trains au départ",
    "Retard moyen de tous les trains à l'arrivée",
    "Retard moyen des trains en retard à l'arrivée",
    "Retard moyen trains en retard > 15 (si liaison concurrencée par vol)",
    "Nombre trains en retard > 30min",
]

for col in cols_non_negative:
    if col in df.columns:
        n_neg = (df[col] < 0).sum()
        if n_neg > 0:
            print(f"  {col}: {n_neg} valeurs négatives → remplacées par 0")
            df[col] = df[col].clip(lower=0)

  Retard moyen de tous les trains au départ: 155 valeurs négatives → remplacées par 0
  Retard moyen de tous les trains à l'arrivée: 110 valeurs négatives → remplacées par 0
  Retard moyen des trains en retard à l'arrivée: 2 valeurs négatives → remplacées par 0
  Retard moyen trains en retard > 15 (si liaison concurrencée par vol): 7 valeurs négatives → remplacées par 0
  Nombre trains en retard > 30min: 42 valeurs négatives → remplacées par 0


### 9.3 Durée de trajet = 0 : NaN puis suppression

In [12]:
# Une durée de trajet à 0 est physiquement impossible → on supprime ces lignes
initial_len = len(df)
df = df[df["Durée moyenne du trajet"] > 0]
print(f"Lignes supprimées car durée de trajet = 0 : {initial_len - len(df)}")

Lignes supprimées car durée de trajet = 0 : 358


### 9.4 Flag causes de retard inconnues

In [13]:
# Quand toutes les colonnes Prct sont à 0, on ne sait pas si c'est 'pas de retard'
# ou 'données non renseignées'. On crée un flag pour distinguer.
pct_cols = [c for c in df.columns if c.startswith('Prct')]

df["causes_connues"] = df[pct_cols].sum(axis=1) > 0

print(f"Lignes avec causes connues   : {df['causes_connues'].sum()}")
print(f"Lignes avec causes inconnues : {(~df['causes_connues']).sum()}")

Lignes avec causes connues   : 10883
Lignes avec causes inconnues : 193


## 10. Export

In [14]:
# Suppression des sauts de ligne dans les commentaires (sinon CSV cassé)
comment_cols_export = [
    "Commentaire annulations",
    "Commentaire retards au départ",
    "Commentaire retards à l'arrivée"
]
for col in comment_cols_export:
    if col in df.columns:
        df[col] = df[col].str.replace(r'[\n\r]+', ' ', regex=True).str.strip()

# Conversion de Date en string pour éviter la perte du format à l'export
df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")

output_file = "cleaned_dataset.csv"
df.to_csv(output_file, index=False, sep=",", encoding="utf-8", quoting=1)

print(f"Dataset nettoyé exporté : {output_file}")
print(f"    Lignes finales : {len(df)}")
df.info()

Dataset nettoyé exporté : cleaned_dataset.csv
    Lignes finales : 11076
<class 'pandas.core.frame.DataFrame'>
Index: 11076 entries, 0 to 12056
Data columns (total 29 columns):
 #   Column                                                                                       Non-Null Count  Dtype  
---  ------                                                                                       --------------  -----  
 0   Date                                                                                         11076 non-null  object 
 1   Service                                                                                      11076 non-null  string 
 2   Gare de départ                                                                               11076 non-null  string 
 3   Gare d'arrivée                                                                               11076 non-null  string 
 4   Durée moyenne du trajet                                                               